In [1]:
import geemap

In [2]:
import ee

In [3]:
ee.Authenticate()

True

In [4]:
ee.Initialize()

In [5]:
# CSV

In [6]:
import ee
import pandas as pd
import numpy as np
import geemap
import ipywidgets as widgets
from IPython.display import display
from ipyleaflet import WidgetControl
import os

# Initialize Earth Engine
ee.Initialize()

# === CONFIGURATION ===
user_folder = '/Users/armindadras/Documents/Fortran_test3/'  # Update if needed
file_indices = [100, 200, 300, 400, 500, 600, 700, 800, 1000]
csv_files = [os.path.join(user_folder, f'matrix_{i:04d}.csv') for i in file_indices]

# === HELPERS ===
def simplify_values(df, precision=1):
    df['value'] = df['value'].round(precision)
    return df

def deduplicate_features(df):
    return df.drop_duplicates(subset=['longitude', 'latitude', 'value'])

def reduce_rows(df):
    return df.iloc[::4].reset_index(drop=True)

def csv_to_feature_collection_half(csv_file, precision=1):
    df = pd.read_csv(csv_file)
    df = reduce_rows(df)
    df = simplify_values(df, precision=precision)
    df = deduplicate_features(df)

    features = []
    for _, row in df.iterrows():
        pt = ee.Geometry.Point([row['longitude'], row['latitude']])
        features.append(ee.Feature(pt, {'value': row['value']}))

    return ee.FeatureCollection(features)

# === LOAD CSVs + CONVERT TO IMAGES ===
images = []
labels = []

for fname in csv_files:
    try:
        fc = csv_to_feature_collection_half(fname)
        img = fc.reduceToImage(['value'], ee.Reducer.first())
        images.append(img)
        labels.append(os.path.basename(fname))
    except Exception as e:
        print(f"Error processing {fname}: {e}")

# === VISUALIZATION SETUP ===
visParams = {
    'min': 0,
    'max': 100,
    'palette': ['green', 'green', 'yellow', 'yellow', 'red', 'red', 'black']
}

# Create the map and add initial image
Map = geemap.Map(center=[52.80, -124.23], zoom=10)
Map.addLayer(images[0], visParams, 'MatrixLayer', opacity=0.5)

# === SLIDER WIDGET ===
slider = widgets.IntSlider(
    min=0,
    max=len(images)-1,
    step=1,
    value=0,
    description='Frame',
    style={'description_width': '60px'},
    layout=widgets.Layout(width='300px')
)

def update_layer(change):
    idx = change['new']
    Map.remove_layer('MatrixLayer')
    Map.addLayer(images[idx], visParams, 'MatrixLayer', opacity=0.99)

slider.observe(update_layer, names='value')

# === ADD SLIDER TO MAP ===
slider_box = widgets.VBox([widgets.Label("Select Matrix Frame:"), slider])
slider_control = WidgetControl(widget=slider_box, position='topright')
Map.add_control(slider_control)

# === DISPLAY MAP ===
display(Map)


Map(center=[52.8, -124.23], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataG…

In [7]:
# CSV to TIFF (not needed)

In [7]:
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.transform import from_origin

# Configuration
input_folder = '/Users/armindadras/Documents/Fortran_test3/'
output_folder = os.path.join(input_folder, 'geotiffs')
# os.makedirs(output_folder, exist_ok=True)

origin_x = -124.23
origin_y = 52.80
pixel_size = 2.5e-3

# Indices for your 10 matrices
file_indices = list(range(100, 1201, 100))

# Process each CSV and convert to GeoTIFF
for idx in file_indices:
    csv_path = os.path.join(input_folder, f'matrix_{idx:04d}.csv')
    tif_path = os.path.join(output_folder, f'matrix_{idx:04d}.tif')

    df = pd.read_csv(csv_path)
    n_pixels = int(np.sqrt(len(df)))
    assert n_pixels * n_pixels == len(df), "CSV does not contain a square grid."

    # Pivot to 2D grid
    df_sorted = df.sort_values(by=['latitude', 'longitude'], ascending=[False, True])
    matrix = df_sorted['value'].values.reshape((n_pixels, n_pixels))

    # Define transform and save as GeoTIFF
    transform = from_origin(origin_x, origin_y, pixel_size, pixel_size)

    with rasterio.open(
        tif_path,
        'w',
        driver='GTiff',
        height=matrix.shape[0],
        width=matrix.shape[1],
        count=1,
        dtype=matrix.dtype,
        crs='EPSG:4326',
        transform=transform
    ) as dst:
        dst.write(matrix, 1)


In [9]:
# Directlt dat to tiff

In [16]:
import os
import numpy as np
import rasterio
from rasterio.transform import from_origin

# Geotransform parameters
origin_x = -124.23
origin_y = 52.80
pixel_size = 2.5e-3

# Input/output directories (adjust as needed)
input_dir = '/Users/armindadras/Documents/Fortran_test3'
output_dir = os.path.join(input_dir, 'geotiffs_from_dat')
os.makedirs(output_dir, exist_ok=True)

# Loop through files from 0100 to 1200 in steps of 100
for num in range(100, 1201, 50):
    dat_path = os.path.join(input_dir, f'matrix_{num:04d}.dat')
    tif_path = os.path.join(output_dir, f'matrix_{num:04d}.tif')

    # Load matrix data
    data = np.loadtxt(dat_path)

    # Determine the shape
    rows, cols = data.shape

    # Create affine transform
    transform = from_origin(origin_x, origin_y, pixel_size, pixel_size)

    # Save as GeoTIFF
    with rasterio.open(
        tif_path,
        'w',
        driver='GTiff',
        height=rows,
        width=cols,
        count=1,
        dtype=data.dtype,
        crs='EPSG:4326',
        transform=transform
    ) as dst:
        dst.write(data, 1)

    print(f"GeoTIFF saved: {tif_path}")


GeoTIFF saved: /Users/armindadras/Documents/Fortran_test3/geotiffs_from_dat/matrix_0100.tif
GeoTIFF saved: /Users/armindadras/Documents/Fortran_test3/geotiffs_from_dat/matrix_0150.tif
GeoTIFF saved: /Users/armindadras/Documents/Fortran_test3/geotiffs_from_dat/matrix_0200.tif
GeoTIFF saved: /Users/armindadras/Documents/Fortran_test3/geotiffs_from_dat/matrix_0250.tif
GeoTIFF saved: /Users/armindadras/Documents/Fortran_test3/geotiffs_from_dat/matrix_0300.tif
GeoTIFF saved: /Users/armindadras/Documents/Fortran_test3/geotiffs_from_dat/matrix_0350.tif
GeoTIFF saved: /Users/armindadras/Documents/Fortran_test3/geotiffs_from_dat/matrix_0400.tif
GeoTIFF saved: /Users/armindadras/Documents/Fortran_test3/geotiffs_from_dat/matrix_0450.tif
GeoTIFF saved: /Users/armindadras/Documents/Fortran_test3/geotiffs_from_dat/matrix_0500.tif
GeoTIFF saved: /Users/armindadras/Documents/Fortran_test3/geotiffs_from_dat/matrix_0550.tif
GeoTIFF saved: /Users/armindadras/Documents/Fortran_test3/geotiffs_from_dat/matr

In [18]:
import os
import geemap
import ipywidgets as widgets
from IPython.display import display
from ipyleaflet import WidgetControl

# === Configuration ===
tiff_folder = '/Users/armindadras/Documents/Fortran_test3/geotiffs_from_dat'
file_indices = list(range(100, 1201, 50))
tif_files = [os.path.join(tiff_folder, f'matrix_{i:04d}.tif') for i in file_indices]

# === Create the interactive map ===
Map = geemap.Map(center=[52.8, -124.23], zoom=10)

# === UI Elements ===
slider = widgets.IntSlider(min=0, max=len(tif_files)-1, step=1, description='Frame')
label = widgets.Label(value=os.path.basename(tif_files[0]))
controls = widgets.VBox([label, slider])
slider_control = WidgetControl(widget=controls, position='topright')
Map.add_control(slider_control)

# === Remove unexpected startup layers ===
for lyr in Map.layers[1:]:
    Map.remove(lyr)

# === Add initial raster layer with correct name ===
last_layer = Map.add_raster(
    tif_files[0],
    layer_name=os.path.basename(tif_files[0]),
    palette=['green', 'yellow', 'red', 'black'],
    vmin=0,
    vmax=100,
    opacity=0.3
)
##########################
from datetime import datetime, timedelta

# Number of frames
num_frames = len(tif_files)

# Starting datetime
start_time = datetime(2024, 2, 1, 0, 0)

# Generate list of timestamp labels
timestamps = [
    (start_time + timedelta(minutes=10 * i)).strftime("%Y-%m-%d %H:%M")
    for i in range(num_frames)
]

########################

# === Update function: clean all non-base layers, then add selected one ===
def update_layer(change):
    global last_layer
    idx = change['new']
    
    # Remove all layers except base map
    for lyr in Map.layers[1:]:
        Map.remove(lyr)

    # Add the new raster layer
    last_layer = Map.add_raster(
        tif_files[idx],
        layer_name=os.path.basename(tif_files[idx]),
        palette=['green', 'yellow', 'red', 'black'],
        vmin=0,
        vmax=100,
        opacity=0.35
    )

    # Update label to show time instead of filename
    label.value = f"Frame {idx} – {timestamps[idx]}"


# Attach the update function to the slider
slider.observe(update_layer, names='value')

# === Show everything ===
display(Map)


Map(center=[52.8, -124.23], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataG…